# Fine-Tune a Generative AI Model for Dialogue Summarization (35 Points)

In this notebook, you will fine-tune an existing LLM from Hugging Face for enhanced dialogue summarization. You will use the [FLAN-T5](https://huggingface.co/docs/transformers/model_doc/flan-t5) model, which provides a high quality instruction tuned model and can summarize text out of the box. To improve the inferences, you will explore a full fine-tuning approach and evaluate the results with ROUGE metrics. Then you will perform Parameter Efficient Fine-Tuning (PEFT), evaluate the resulting model and see that the benefits of PEFT outweigh the slightly-lower performance metrics.

# Table of Contents

- [ 1 - Load Required Dependencies, Dataset and LLM](#1)
  - [ 1.1 - Set up Required Dependencies](#1.1)
  - [ 1.2 - Load Dataset and LLM](#1.2)
  - [ 1.3 - Test the Model with Zero Shot Inferencing](#1.3)
- [ 2 - Perform Full Fine-Tuning](#2)
  - [ 2.1 - Preprocess the Dialog-Summary Dataset](#2.1)
  - [ 2.2 - Fine-Tune the Model with the Preprocessed Dataset](#2.2)
  - [ 2.3 - Evaluate the Model Qualitatively (Human Evaluation)](#2.3)
  - [ 2.4 - Evaluate the Model Quantitatively (with ROUGE Metric)](#2.4)
- [ 3 - Perform Parameter Efficient Fine-Tuning (PEFT)](#3)
  - [ 3.1 - Setup the PEFT/LoRA model for Fine-Tuning](#3.1)
  - [ 3.2 - Train PEFT Adapter](#3.2)
  - [ 3.3 - Evaluate the Model Qualitatively (Human Evaluation)](#3.3)
  - [ 3.4 - Evaluate the Model Quantitatively (with ROUGE Metric)](#3.4)

<a name='1'></a>
## 1 - Load Required Dependencies, Dataset and LLM (10 points)

<a name='1.1'></a>
### 1.1 - Set up Required Dependencies (2 point)

Now install the required packages for the LLM and datasets.



In [1]:
!pip install -q datasets torch transformers evaluate rouge_score loralib peft


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 which is incompatible.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Import the necessary components. Some of them are new for this week, they will be discussed later in the notebook.

In [2]:
import os
import time

import evaluate
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float32

WORKDIR = "."

FULL_FT_CHECKPOINT_CANDIDATES = [
    os.path.join(WORKDIR, checkpoint_name)
    for checkpoint_name in ["flan-diaglogue-summary-checkpoint", "flan-dialogue-summary-checkpoint"]
]

PEFT_CHECKPOINT_CANDIDATES = [
    os.path.join(WORKDIR, checkpoint_name)
    for checkpoint_name in ["peft-dialogue-summary-checkpoint-from-s3", "peft-dialogue-summary-checkpoint-local"]
]

RESULTS_CSV = os.path.join(WORKDIR, "dialogue-summary-training-results.csv")


def pick_path(candidates):
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]


def load_seq2seq_model(model_path):
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path, torch_dtype=model_dtype)
    return model.to(device)


def build_prompt(dialogue):
    return f"""Summarize the following conversation.

{dialogue}

Summary: """


def generate_summary(model, prompt, max_new_tokens=200):
    input_ids = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).input_ids.to(device)
    outputs = model.generate(input_ids=input_ids, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<a name='1.2'></a>
### 1.2 - Load Dataset and LLM (4 points)

You are going to continue experimenting with the [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) Hugging Face dataset. It contains 10,000+ dialogues with the corresponding manually labeled summaries and topics.

In [3]:
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

dataset


Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 12460/12460 [00:00<00:00, 120365.25 examples/s]

Generating train split: 100%|██████████| 12460/12460 [00:00<00:00, 119544.22 examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating validation split: 100%|██████████| 500/500 [00:00<00:00, 93794.53 examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split: 100%|██████████| 1500/1500 [00:00<00:00, 164779.76 examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

Load the pre-trained [FLAN-T5 model](https://huggingface.co/docs/transformers/model_doc/flan-t5) and its tokenizer directly from HuggingFace. Notice that you will be using the [small version](https://huggingface.co/google/flan-t5-small) of FLAN-T5. Setting `torch_dtype=torch.bfloat16` specifies the memory type to be used by this model.

In [4]:
model_name = "google/flan-t5-small"

original_model = load_seq2seq_model(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 10558.98it/s]


The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


It is possible to pull out the number of model parameters and find out how many of them are trainable. The following function can be used to do that, at this stage, you do not need to go into details of it.

In [5]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0

    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()

    return (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: "
        f"{100 * trainable_model_params / all_model_params:.2f}%"
    )


print(print_number_of_trainable_model_parameters(original_model))


trainable model parameters: 76961152
all model parameters: 76961152
percentage of trainable model parameters: 100.00%


<a name='1.3'></a>
### 1.3 - Test the Model with Zero Shot Inferencing (2 Points)

Test the model with the zero shot inferencing. You can see that the model struggles to summarize the dialogue compared to the baseline summary, but it does pull out some important information from the text which indicates the model can be fine-tuned to the task at hand.

In [6]:
index = 200

dialogue = dataset["test"][index]["dialogue"]
summary = dataset["test"][index]["summary"]

prompt = build_prompt(dialogue)
output = generate_summary(original_model, prompt)

dash_line = "-".join("" for x in range(100))
print(dash_line)
print(f"INPUT PROMPT:\n{prompt}")
print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}\n")
print(dash_line)
print(f"MODEL GENERATION - ZERO SHOT:\n{output}")


---------------------------------------------------------------------------------------------------
INPUT PROMPT:
Summarize the following conversation.

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

Summary: 
--------------------------------------------------------------------

<a name='2'></a>
## 2 - Perform Full Fine-Tuning (15 points)

<a name='2.1'></a>
### 2.1 - Preprocess the Dialog-Summary Dataset (3 points)

You need to convert the dialog-summary (prompt-response) pairs into explicit instructions for the LLM. Prepend an instruction to the start of the dialog with `Summarize the following conversation` and to the start of the summary with `Summary` as follows:

Training prompt (dialogue):
```
Summarize the following conversation.

    Chris: This is his part of the conversation.
    Antje: This is her part of the conversation.
    
Summary:
```

Training response (summary):
```
Both Chris and Antje participated in the conversation.
```

Then preprocess the prompt-response dataset into tokens and pull out their `input_ids` (1 per token).

In [7]:
def tokenize_function(example):
    prompts = [build_prompt(dialogue) for dialogue in example["dialogue"]]
    model_inputs = tokenizer(
        prompts,
        max_length=512,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=example["summary"],
        max_length=128,
        truncation=True,
        padding="max_length",
    )

    model_inputs["labels"] = [
        [token if token != tokenizer.pad_token_id else -100 for token in label]
        for label in labels["input_ids"]
    ]
    return model_inputs


# The dataset actually contains 3 diff splits: train, validation, test.
# The tokenize_function code is handling all data across all splits in batches.
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(dataset["train"].column_names)
tokenized_datasets.set_format("torch")


Map:   0%|          | 0/12460 [00:00<?, ? examples/s]

Map:   8%|▊         | 1000/12460 [00:00<00:02, 5190.35 examples/s]

Map:  16%|█▌        | 2000/12460 [00:00<00:02, 5148.37 examples/s]

Map:  24%|██▍       | 3000/12460 [00:00<00:01, 5128.98 examples/s]

Map:  32%|███▏      | 4000/12460 [00:00<00:01, 4660.59 examples/s]

...

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map: 100%|██████████| 500/500 [00:00<00:00, 5419.70 examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:  67%|██████▋   | 1000/1500 [00:00<00:00, 5376.42 examples/s]

Map: 100%|██████████| 1500/1500 [00:00<00:00, 5189.33 examples/s]

To save some time in the lab, you will subsample the dataset:

In [8]:
tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 100 == 0, with_indices=True)

Filter:   0%|          | 0/12460 [00:00<?, ? examples/s]

Filter: 100%|██████████| 12460/12460 [00:00<00:00, 234718.57 examples/s]

Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

Filter: 100%|██████████| 500/500 [00:00<00:00, 120132.44 examples/s]

Filter:   0%|          | 0/1500 [00:00<?, ? examples/s]

Filter: 100%|██████████| 1500/1500 [00:00<00:00, 187362.82 examples/s]

Check the shapes of all three parts of the dataset:

In [9]:
print(f"Shapes of the datasets:")
print(f"Training: {tokenized_datasets['train'].shape}")
print(f"Validation: {tokenized_datasets['validation'].shape}")
print(f"Test: {tokenized_datasets['test'].shape}")

print(tokenized_datasets)

Shapes of the datasets:
Training: (125, 3)
Validation: (5, 3)
Test: (15, 3)
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 125
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 15
    })
})


The output dataset is ready for fine-tuning.

<a name='2.2'></a>
### 2.2 - Fine-Tune the Model with the Preprocessed Dataset (4 points)

Now utilize the built-in Hugging Face `Trainer` class (see the documentation [here](https://huggingface.co/docs/transformers/main_classes/trainer)). Pass the preprocessed dataset with reference to the original model. Other training parameters are found experimentally and there is no need to go into details about those at the moment.

In [10]:
output_dir = f"./dialogue-summary-training-{str(int(time.time()))}"

training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-5,
    num_train_epochs=1,
    logging_steps=1,
    max_steps=1,
    report_to="none",
)

trainer = Trainer(
    model=original_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)


Start training process...



The code trainer.train() utilizes the Weights & Biases (wandb) library to track and visualize the training process. To proceed, you'll need to sign up for a wandb account using your Gmail and then enter your unique API token to authenticate and enable logging of the training progress.

In [11]:
trainer.train()

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Saved the 1-step fine-tuning checkpoint to: {output_dir}")
print("Later evaluation cells will prefer the provided Drive checkpoint when it is available.")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,2.858405


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.54it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]

Saved the 1-step fine-tuning checkpoint to: ./dialogue-summary-training-1775959528
Later evaluation cells will prefer the provided Drive checkpoint when it is available.


Create an instance of the `AutoModelForSeq2SeqLM` class for the instruct model:

In [12]:
# Keep any provided checkpoints in subdirectories of the current working directory.
# Expected local names are shown below so later cells can load them without extra path edits.
print("Current working directory:", os.getcwd())
print("Full fine-tuned checkpoint candidates:", FULL_FT_CHECKPOINT_CANDIDATES)
print("PEFT checkpoint candidates:", PEFT_CHECKPOINT_CANDIDATES)
print("Results CSV:", RESULTS_CSV)


Current working directory: /Users/anthony/Documents/DS/DS-301/HW4
Full fine-tuned checkpoint candidates: ['./flan-diaglogue-summary-checkpoint', './flan-dialogue-summary-checkpoint']
PEFT checkpoint candidates: ['./peft-dialogue-summary-checkpoint-from-s3', './peft-dialogue-summary-checkpoint-local']
Results CSV: ./dialogue-summary-training-results.csv


In [13]:
# Load tokenizer and models
model_path = pick_path(FULL_FT_CHECKPOINT_CANDIDATES + [output_dir])

tokenizer = AutoTokenizer.from_pretrained(model_name)
original_model = load_seq2seq_model(model_name)
instruct_model = load_seq2seq_model(model_path)

print(f"Using full fine-tuned checkpoint: {model_path}")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 6524.52it/s]


The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 6826.73it/s]


The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Using full fine-tuned checkpoint: ./dialogue-summary-training-1775959528


<a name='2.3'></a>
### 2.3 - Evaluate the Model Qualitatively (Human Evaluation) (4 points)

As with many GenAI applications, a qualitative approach where you ask yourself the question "Is my model behaving the way it is supposed to?" is usually a good starting point. In the example below (the same one we started this notebook with), you can see how the fine-tuned model is able to create a reasonable summary of the dialogue compared to the original inability to understand what is being asked of the model.

In [14]:
index = 200
dialogue = dataset["test"][index]["dialogue"]
human_baseline_summary = dataset["test"][index]["summary"]

prompt = build_prompt(dialogue)

original_model_text_output = generate_summary(original_model, prompt)
instruct_model_text_output = generate_summary(instruct_model, prompt)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{human_baseline_summary}")
print(dash_line)
print(f"ORIGINAL MODEL:\n{original_model_text_output}")
print(dash_line)
print(f"INSTRUCT MODEL:\n{instruct_model_text_output}")


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
How would you like to upgrade your computer?
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL:
How would you like to upgrade your computer?


<a name='2.4'></a>
### 2.4 - Evaluate the Model Quantitatively (with ROUGE Metric) (4 points)

The [ROUGE metric](https://en.wikipedia.org/wiki/ROUGE_(metric)) helps quantify the validity of summarizations produced by models. It compares summarizations to a "baseline" summary which is usually created by a human. While not perfect, it does indicate the overall increase in summarization effectiveness that we have accomplished by fine-tuning.

In [15]:
rouge = evaluate.load("rouge")


Generate the outputs for the sample of the test dataset (only 10 dialogues and summaries to save time), and save the results.

In [16]:
dialogues = dataset["test"][0:10]["dialogue"]
human_baseline_summaries = dataset["test"][0:10]["summary"]

original_model_summaries = []
instruct_model_summaries = []

for dialogue in dialogues:
    prompt = build_prompt(dialogue)
    original_model_summaries.append(generate_summary(original_model, prompt))
    instruct_model_summaries.append(generate_summary(instruct_model, prompt))

zipped_summaries = list(
    zip(human_baseline_summaries, original_model_summaries, instruct_model_summaries)
)

df = pd.DataFrame(
    zipped_summaries,
    columns=["human_baseline_summaries", "original_model_summaries", "instruct_model_summaries"],
)
df


,human_baseline_summaries,original_model_summaries,instruct_model_summaries
0,Ms. Dawson helps #Person1# to write a memo to ...,Is this all correct?,Is this all correct?
1,In order to prevent employees from wasting tim...,Is this all correct?,Is this all correct?
2,Ms. Dawson takes a dictation for #Person1# abo...,Is this all correct?,Is this all correct?
3,#Person2# arrives late because of traffic jam....,Talk to the driver.,Talk to the driver.
4,#Person2# decides to follow #Person1#'s sugges...,Talk to the driver.,Talk to the driver.
5,#Person2# complains to #Person1# about the tra...,Talk to the driver.,Talk to the driver.
6,#Person1# tells Kate that Masha and Hero get d...,M: I think it's a good idea to have a couple o...,M: I think it's a good idea to have a couple o...
7,#Person1# tells Kate that Masha and Hero are g...,M: I think it's a good idea to have a couple o...,M: I think it's a good idea to have a couple o...
8,#Person1# and Kate talk about the divorce betw...,M: I think it's a good idea to have a couple o...,M: I think it's a good idea to have a couple o...
9,#Person1# and Brian are at the birthday party ...,"Brian, how are you?","Brian, how are you?"


Evaluate the models computing ROUGE metrics. Notice the improvement in the results!

In [17]:
original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)


ORIGINAL MODEL:
{'rouge1': 0.08509376386614238, 'rouge2': 0.0, 'rougeL': 0.08418921232220464, 'rougeLsum': 0.0865154155307608}
INSTRUCT MODEL:
{'rouge1': 0.08509376386614238, 'rouge2': 0.0, 'rougeL': 0.08418921232220464, 'rougeLsum': 0.0865154155307608}


The file `dialogue-summary-training-results.csv` contains a pre-populated list of all model results which you can use to evaluate on a larger section of data. The next cell reads that file directly from the current working directory.


In [18]:
results_path = RESULTS_CSV
results = pd.read_csv(results_path).drop(columns=["Unnamed: 0"], errors="ignore")

human_baseline_summaries = results["human_baseline_summaries"].values
original_model_summaries = results["original_model_summaries"].values
instruct_model_summaries = results["instruct_model_summaries"].values

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

print(f"Loaded results from: {results_path}")
print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)


Loaded results from: ./dialogue-summary-training-results.csv
ORIGINAL MODEL:
{'rouge1': 0.2216686882994889, 'rouge2': 0.0707492488737373, 'rougeL': 0.19245630286595683, 'rougeLsum': 0.192409231638204}
INSTRUCT MODEL:
{'rouge1': 0.4041959932817219, 'rouge2': 0.17064828985299663, 'rougeL': 0.3267557101191949, 'rougeLsum': 0.3266766725171105}


The results show substantial improvement in all ROUGE metrics:

In [19]:
print("Absolute percentage improvement of INSTRUCT MODEL over ORIGINAL MODEL")

improvement = (np.array(list(instruct_model_results.values())) - np.array(list(original_model_results.values())))
for key, value in zip(instruct_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of INSTRUCT MODEL over ORIGINAL MODEL
rouge1: 18.25%
rouge2: 9.99%
rougeL: 13.43%
rougeLsum: 13.43%


<a name='3'></a>
## 3 - Perform Parameter Efficient Fine-Tuning (PEFT) (10 points)

Now, let's perform **Parameter Efficient Fine-Tuning (PEFT)** fine-tuning as opposed to "full fine-tuning" as you did above. PEFT is a form of instruction fine-tuning that is much more efficient than full fine-tuning - with comparable evaluation results as you will see soon.

PEFT is a generic term that includes **Low-Rank Adaptation (LoRA)** and prompt tuning (which is NOT THE SAME as prompt engineering!). In most cases, when someone says PEFT, they typically mean LoRA. LoRA, at a very high level, allows the user to fine-tune their model using fewer compute resources (in some cases, a single GPU). After fine-tuning for a specific task, use case, or tenant with LoRA, the result is that the original LLM remains unchanged and a newly-trained “LoRA adapter” emerges. This LoRA adapter is much, much smaller than the original LLM - on the order of a single-digit % of the original LLM size (MBs vs GBs).  

That said, at inference time, the LoRA adapter needs to be reunited and combined with its original LLM to serve the inference request.  The benefit, however, is that many LoRA adapters can re-use the original LLM which reduces overall memory requirements when serving multiple tasks and use cases.

<a name='3.1'></a>
### 3.1 - Setup the PEFT/LoRA model for Fine-Tuning (2 points)

You need to set up the PEFT/LoRA model for fine-tuning with a new layer/parameter adapter. Using PEFT/LoRA, you are freezing the underlying LLM and only training the adapter. Have a look at the LoRA configuration below. Note the rank (`r`) hyper-parameter, which defines the rank/dimension of the adapter to be trained.

In [20]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM,  # FLAN-T5
)


Add LoRA adapter layers/parameters to the original LLM to be trained.

In [21]:
peft_model = get_peft_model(load_seq2seq_model(model_name), lora_config)
print(print_number_of_trainable_model_parameters(peft_model))


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 61291.94it/s]


The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable model parameters: 1376256
all model parameters: 78337408
percentage of trainable model parameters: 1.76%


<a name='3.2'></a>
### 3.2 - Train PEFT Adapter (3 points)

Define training arguments and create `Trainer` instance.

In [22]:
output_dir = f"./peft-dialogue-summary-training-{str(int(time.time()))}"

peft_training_args = TrainingArguments(
    output_dir=output_dir,
    auto_find_batch_size=True,
    learning_rate=1e-3,
    num_train_epochs=1,
    logging_steps=1,
    max_steps=1,
    report_to="none",
)

peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
)


Now everything is ready to train the PEFT adapter and save the model.



In [23]:
peft_trainer.train()

peft_model_path = "./peft-dialogue-summary-checkpoint-local"

peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)

print(f"Saved the 1-step PEFT adapter checkpoint to: {peft_model_path}")
print("Later evaluation cells will prefer the provided Drive PEFT checkpoint when it is available.")


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
1,3.005809


Saved the 1-step PEFT adapter checkpoint to: ./peft-dialogue-summary-checkpoint-local
Later evaluation cells will prefer the provided Drive PEFT checkpoint when it is available.


That training was performed on a subset of data. To load a fully trained PEFT model, place the PEFT checkpoint directory inside the current working directory before running the next cell.

Prepare this model by adding an adapter to the original FLAN-T5 model. You are setting `is_trainable=False` because the plan is only to perform inference with this PEFT model. If you were preparing the model for further training, you would set `is_trainable=True`.

In [24]:
from peft import PeftModel, PeftConfig

peft_model_base = load_seq2seq_model(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

peft_model_path = pick_path(PEFT_CHECKPOINT_CANDIDATES + ["./peft-dialogue-summary-checkpoint-local"])

peft_model = PeftModel.from_pretrained(
    peft_model_base,
    peft_model_path,
    is_trainable=False,
)

peft_model = peft_model.to(device)

print(f"Using PEFT checkpoint: {peft_model_path}")


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 23990.54it/s]


The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Using PEFT checkpoint: ./peft-dialogue-summary-checkpoint-local


The number of trainable parameters will be `0` due to `is_trainable=False` setting:

In [25]:
print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 0
all model parameters: 78337408
percentage of trainable model parameters: 0.00%


<a name='3.3'></a>
### 3.3 - Evaluate the Model Qualitatively (Human Evaluation) (2 points)

Make inferences for the same example as in sections [1.3](#1.3) and [2.3](#2.3), with the original model, fully fine-tuned and PEFT model.

In [26]:
index = 200
dialogue = dataset["test"][index]["dialogue"]
human_baseline_summary = dataset["test"][index]["summary"]

prompt = build_prompt(dialogue)

original_model_text_output = generate_summary(original_model, prompt)
instruct_model_text_output = generate_summary(instruct_model, prompt)
peft_model_text_output = generate_summary(peft_model, prompt)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{human_baseline_summary}")
print(dash_line)
print(f"ORIGINAL MODEL:\n{original_model_text_output}")
print(dash_line)
print(f"INSTRUCT MODEL:\n{instruct_model_text_output}")
print(dash_line)
print(f"PEFT MODEL:\n{peft_model_text_output}")


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.
---------------------------------------------------------------------------------------------------
ORIGINAL MODEL:
How would you like to upgrade your computer?
---------------------------------------------------------------------------------------------------
INSTRUCT MODEL:
How would you like to upgrade your computer?
---------------------------------------------------------------------------------------------------
PEFT MODEL:
How would you like to upgrade your computer?


<a name='3.4'></a>
### 3.4 - Evaluate the Model Quantitatively (with ROUGE Metric) (3 points)
Perform inferences for the sample of the test dataset (only 10 dialogues and summaries to save time).

In [27]:
dialogues = dataset["test"][0:10]["dialogue"]
human_baseline_summaries = dataset["test"][0:10]["summary"]

original_model_summaries = []
instruct_model_summaries = []
peft_model_summaries = []

for dialogue in dialogues:
    prompt = build_prompt(dialogue)

    original_model_summaries.append(generate_summary(original_model, prompt))
    instruct_model_summaries.append(generate_summary(instruct_model, prompt))
    peft_model_summaries.append(generate_summary(peft_model, prompt))

zipped_summaries = list(
    zip(
        human_baseline_summaries,
        original_model_summaries,
        instruct_model_summaries,
        peft_model_summaries,
    )
)

df = pd.DataFrame(
    zipped_summaries,
    columns=[
        "human_baseline_summaries",
        "original_model_summaries",
        "instruct_model_summaries",
        "peft_model_summaries",
    ],
)
df


,human_baseline_summaries,original_model_summaries,instruct_model_summaries,peft_model_summaries
0,Ms. Dawson helps #Person1# to write a memo to ...,Is this all correct?,Is this all correct?,Is this all correct?
1,In order to prevent employees from wasting tim...,Is this all correct?,Is this all correct?,Is this all correct?
2,Ms. Dawson takes a dictation for #Person1# abo...,Is this all correct?,Is this all correct?,Is this all correct?
3,#Person2# arrives late because of traffic jam....,Talk to the driver.,Talk to the driver.,Talk to the driver.
4,#Person2# decides to follow #Person1#'s sugges...,Talk to the driver.,Talk to the driver.,Talk to the driver.
5,#Person2# complains to #Person1# about the tra...,Talk to the driver.,Talk to the driver.,Talk to the driver.
6,#Person1# tells Kate that Masha and Hero get d...,M: I think it's a good idea to have a couple o...,M: I think it's a good idea to have a couple o...,"Kate, you never believe what happened."
7,#Person1# tells Kate that Masha and Hero are g...,M: I think it's a good idea to have a couple o...,M: I think it's a good idea to have a couple o...,"Kate, you never believe what happened."
8,#Person1# and Kate talk about the divorce betw...,M: I think it's a good idea to have a couple o...,M: I think it's a good idea to have a couple o...,"Kate, you never believe what happened."
9,#Person1# and Brian are at the birthday party ...,"Brian, how are you?","Brian, how are you?","Brian, how are you?"


In [28]:
rouge = evaluate.load("rouge")

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries,
)

print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)
print("PEFT MODEL:")
print(peft_model_results)


ORIGINAL MODEL:
{'rouge1': 0.08509376386614238, 'rouge2': 0.0, 'rougeL': 0.08418921232220464, 'rougeLsum': 0.0865154155307608}
INSTRUCT MODEL:
{'rouge1': 0.08509376386614238, 'rouge2': 0.0, 'rougeL': 0.08418921232220464, 'rougeLsum': 0.0865154155307608}
PEFT MODEL:
{'rouge1': 0.09070628244541287, 'rouge2': 0.0, 'rougeL': 0.08956493828232959, 'rougeLsum': 0.09084212084212084}


Notice, that PEFT model results are not too bad, while the training process was much easier!

You already computed ROUGE score on the full dataset using `dialogue-summary-training-results.csv` from the current working directory. The next cell reuses that dataframe to compare the original, full fine-tuned, and PEFT models on the larger saved result set.


In [29]:
human_baseline_summaries = results["human_baseline_summaries"].values
original_model_summaries = results["original_model_summaries"].values
instruct_model_summaries = results["instruct_model_summaries"].values
peft_model_summaries = results["peft_model_summaries"].values

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries,
)

instruct_model_results = rouge.compute(
    predictions=instruct_model_summaries,
    references=human_baseline_summaries,
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries,
)

print("ORIGINAL MODEL:")
print(original_model_results)
print("INSTRUCT MODEL:")
print(instruct_model_results)
print("PEFT MODEL:")
print(peft_model_results)


ORIGINAL MODEL:
{'rouge1': 0.2216686882994889, 'rouge2': 0.0707492488737373, 'rougeL': 0.19245630286595683, 'rougeLsum': 0.192409231638204}
INSTRUCT MODEL:
{'rouge1': 0.4041959932817219, 'rouge2': 0.17064828985299663, 'rougeL': 0.3267557101191949, 'rougeLsum': 0.3266766725171105}
PEFT MODEL:
{'rouge1': 0.39119098357131776, 'rouge2': 0.15459808342905274, 'rougeL': 0.31367299251500014, 'rougeLsum': 0.31360615168633016}


The results show less of an improvement over full fine-tuning, but the benefits of PEFT typically outweigh the slightly-lower performance metrics.

Calculate the improvement of PEFT over the original model:

In [30]:
print("Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(original_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of PEFT MODEL over ORIGINAL MODEL
rouge1: 16.95%
rouge2: 8.38%
rougeL: 12.12%
rougeLsum: 12.12%


Now calculate the improvement of PEFT over a full fine-tuned model:

In [31]:
print("Absolute percentage improvement of PEFT MODEL over INSTRUCT MODEL")

improvement = (np.array(list(peft_model_results.values())) - np.array(list(instruct_model_results.values())))
for key, value in zip(peft_model_results.keys(), improvement):
    print(f'{key}: {value*100:.2f}%')

Absolute percentage improvement of PEFT MODEL over INSTRUCT MODEL
rouge1: -1.30%
rouge2: -1.61%
rougeL: -1.31%
rougeLsum: -1.31%


Here you should see a small ROUGE decrease for PEFT versus the full fine-tuned model, while still retaining a clear improvement over the original zero-shot baseline. That trade-off is the main takeaway for the written discussion: PEFT gives up a little quality in exchange for a much cheaper training setup.
